In [2]:
from pathlib import Path
import os


In [3]:
BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
ahc_enhanced_inference_labels_dir =Path(os.path.join(BASE_DIR, '/workspace/nnUNet_raw/Dataset301_RACLETTE_4DFLOW_AORTA')) / "labelsTs"
labels_dir = sorted(list(ahc_enhanced_inference_labels_dir.glob("*.nii.gz")))
ahc_enhanced_inference_images_dir =Path(os.path.join(BASE_DIR, '/workspace/4dflows_magnitude_ahc')) / "imagesTs_enhanced"
enhanced_images_dir = sorted(list(ahc_enhanced_inference_images_dir.glob("*.nii.gz")))


In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import numpy as np

# Ensure images and labels are aligned
assert len(enhanced_images_dir) == len(labels_dir)

for img_path, lbl_path in zip(enhanced_images_dir, labels_dir):

    # Read image and label
    image = sitk.GetArrayFromImage(sitk.ReadImage(str(img_path)))
    label = sitk.GetArrayFromImage(sitk.ReadImage(str(lbl_path)))

    print(f"Image shape: {image.shape}, Label shape: {label.shape}")

    # Handle 4D image (e.g. time,z,y,x)
    if image.ndim == 4:
        image = image[image.shape[0] // 2]  # middle time frame

    # Handle 4D label if needed
    if label.ndim == 4:
        label = label[label.shape[0] // 2]

    # Middle slice
    z = image.shape[0] // 2

    plt.figure(figsize=(8, 8))
    plt.imshow(image[z], cmap="gray")
    plt.imshow(
        np.ma.masked_where(label[z] == 0, label[z]),
        cmap="jet",
        alpha=0.4
    )
    plt.title(img_path.stem)
    plt.axis("off")
    plt.show()

In [ ]:

def keep_largest_component_per_label(seg):
    """
    Keep only the largest connected component for each label.
    Background (0) is preserved.
    """
    seg_np = sitk.GetArrayFromImage(seg)

    labels = sorted(set(seg_np.flatten()))
    labels = [l for l in labels if l != 0]

    output = sitk.Image(seg.GetSize(), sitk.sitkUInt8)
    output.CopyInformation(seg)

    for label in labels:
        binary = seg == int(label)

        cc = sitk.ConnectedComponent(binary)

        stats = sitk.LabelShapeStatisticsImageFilter()
        stats.Execute(cc)

        if len(stats.GetLabels()) == 0:
            continue

        largest_cc = max(
            stats.GetLabels(),
            key=lambda x: stats.GetPhysicalSize(x)
        )

        largest_mask = cc == largest_cc

        output = output + sitk.Cast(largest_mask, sitk.sitkUInt8) * int(label)

    return output


BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
ahc_enhanced_inference_labels_dir =Path(os.path.join(BASE_DIR, '/workspace/nnUNet_raw/Dataset301_RACLETTE_4DFLOW_AORTA')) / "labelsTs"
ahc_enhanced_inference_labels_lcc_dir =Path(os.path.join(BASE_DIR, '/workspace/4dflows_magnitude_ahc')) / "labelsTs_enhanced_lcc" # inferenced labels after keeping largest connected component for enahnced images

os.makedirs(ahc_enhanced_inference_labels_lcc_dir, exist_ok=True)

for fname in os.listdir(ahc_enhanced_inference_labels_dir):

    if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
        continue

    in_file = os.path.join(ahc_enhanced_inference_labels_dir, fname)
    out_file = os.path.join(ahc_enhanced_inference_labels_lcc_dir, fname)

    print(f"Processing: {fname}")

    seg = sitk.ReadImage(in_file)

    cleaned = keep_largest_component_per_label(seg)

    sitk.WriteImage(cleaned, out_file)

print("Done.")

In [8]:
BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
ahc_enhanced_inference_labels_lcc_dir =Path(os.path.join(BASE_DIR, '/workspace/4dflows_magnitude_ahc')) / "labelsTs_enhanced_lcc" # inferenced labels after keeping largest connected component for enahnced images
labels_dir = sorted(list(ahc_enhanced_inference_labels_lcc_dir.glob("*.nii.gz")))
ahc_enhanced_inference_images_dir =Path(os.path.join(BASE_DIR, '/workspace/4dflows_magnitude_ahc')) / "imagesTs_enhanced"
enhanced_images_dir = sorted(list(ahc_enhanced_inference_images_dir.glob("*.nii.gz")))

In [ ]:
# import SimpleITK as sitk
# import matplotlib.pyplot as plt
# import numpy as np

# # Ensure images and labels are aligned
# assert len(enhanced_images_dir) == len(labels_dir)

# for img_path, lbl_path in zip(enhanced_images_dir, labels_dir):

#     # Read image and label
#     image = sitk.GetArrayFromImage(sitk.ReadImage(str(img_path)))
#     label = sitk.GetArrayFromImage(sitk.ReadImage(str(lbl_path)))

#     print(f"Image shape: {image.shape}, Label shape: {label.shape}")

#     # Handle 4D image (e.g. time,z,y,x)
#     if image.ndim == 4:
#         image = image[image.shape[0] // 2]  # middle time frame

#     # Handle 4D label if needed
#     if label.ndim == 4:
#         label = label[label.shape[0] // 2]

#     # Middle slice
#     z = image.shape[0] // 2

#     plt.figure(figsize=(8, 8))
#     plt.imshow(image[z], cmap="gray")
#     plt.imshow(
#         np.ma.masked_where(label[z] == 0, label[z]),
#         cmap="jet",
#         alpha=0.4
#     )
#     plt.title(img_path.stem)
#     plt.axis("off")
#     plt.show()